# GAN-BERT.
## Part4 - G1 



In [1]:
# !pip install pandas 

In [2]:
!pip install gdown

In [3]:
# import pandas

In [4]:
# Import libraries
import io
import json
import time
import tqdm
import math
import torch
import random
import datetime

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModel, AutoTokenizer, AutoConfig
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

In [5]:
# Set seed
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed_val)

In [6]:
# Check available device
if torch.cuda.is_available():
    
    # Set device
    device = torch.device("cuda")

else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

In [7]:
# Load and Read Json files and create Pandas dataframe
# !gdown 1oh9c-d0fo3NtETNySmCNLUc6H1j4dSWE
# !gdown 1k5LMwmYF7PF-BzYQNE2ULBae79nbM268

!gdown 1LFeGWL49PX5JujrQ2zMbSgxuAFA-Xmbf
!gdown 1SZNvp_hYVczqe0rM1AZKu9tXTE7qHN1j


# Load data and make list of texts and their labels
all_texts_train=[]
all_labels_train=[]
with open('subtaskB_train.jsonl','r') as f:
     for line in f:
        data = json.loads(line)
        all_texts_train.append(data['text'])
        all_labels_train.append(data['model'])

# Load test/vel data
all_texts_test=[]
all_labels_test=[]
lennns=[]
with open('subtaskB_dev.jsonl','r') as f:
    for line in f:
        data = json.loads(line)
        all_texts_test.append(data['text'])
        all_labels_test.append(data['model'])
        lennns.append(len(data['text']))

Downloading...
From: https://drive.google.com/uc?id=1LFeGWL49PX5JujrQ2zMbSgxuAFA-Xmbf
To: /kaggle/working/subtaskB_dev.jsonl
100%|███████████████████████████████████████| 4.93M/4.93M [00:00<00:00, 214MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1SZNvp_hYVczqe0rM1AZKu9tXTE7qHN1j
From (redirected): https://drive.google.com/uc?id=1SZNvp_hYVczqe0rM1AZKu9tXTE7qHN1j&confirm=t&uuid=1efda4b0-abe9-4160-b77c-5708ce9ad9b7
To: /kaggle/working/subtaskB_train.jsonl
100%|█████████████████████████████████████████| 155M/155M [00:00<00:00, 206MB/s]


In [8]:
# Convert to dataframes
df_train = pd.DataFrame({"Text": all_texts_train, "Label": all_labels_train})
df_test = pd.DataFrame({"Text": all_texts_test, "Label": all_labels_test})
df_train.head()

,Text,Label
0,Forza Motorsport is a popular racing game that...,chatGPT
1,Buying Virtual Console games for your Nintendo...,chatGPT
2,Windows NT 4.0 was a popular operating system ...,chatGPT
3,How to Make Perfume\n\nPerfume is a great way ...,chatGPT
4,How to Convert Song Lyrics to a Song'\n\nConve...,chatGPT


In [9]:
# Print Dataset stats
print('Number of datapoints in each class:')
print('Train set:')
print(df_train['Label'].value_counts())
print('*' * 30)
print('Test set:')
print(df_test['Label'].value_counts())
print('*' * 30)
print('Train set Shape:',df_train.shape)
print('Test set Shape:',df_test.shape)

Number of datapoints in each class:
Train set:
Label
davinci    11999
bloomz     11998
human      11997
chatGPT    11995
dolly      11702
cohere     11336
Name: count, dtype: int64
******************************
Test set:
Label
chatGPT    500
human      500
davinci    500
cohere     500
bloomz     500
dolly      500
Name: count, dtype: int64
******************************
Train set Shape: (71027, 2)
Test set Shape: (3000, 2)


In [10]:
# Dataset parameters
max_seq_length = 256
batch_size = 32

# Use P% of the labeled data for training
P = 0.5
df_train_for_ganbert = df_train.sample(frac = P)

# Use the (1 - P) remaining as unlabeled
df_unlabeled = df_train.drop(df_train_for_ganbert.index)

# Print labeled and unlabeled datasets shape
print(df_train_for_ganbert.shape, df_unlabeled.shape)

# available labels in dataset
label_list = ['UNK', 'chatGPT', 'human', 'cohere', 'davinci', 'bloomz', 'dolly']

# Set  unknowne label for unlabeled data
for i in df_unlabeled.index :
    df_unlabeled.at[i, "Label"]= "UNK"

# Show final unlabeled dataset head
df_unlabeled.head()

(35514, 2) (35513, 2)


,Text,Label
2,Windows NT 4.0 was a popular operating system ...,UNK
5,How to Fix a Broken Window in a Wooden Frame\n...,UNK
8,Teaching your dog new tricks is a great way to...,UNK
10,How to Make a Photo Page Using Inkscape\n\nDo ...,UNK
12,How to Stop Fighting With Your Brother or Sist...,UNK


In [11]:
# A function for get examples from df to pass to the dataloader
def get_examples(df):

    # A list to store the datapoints
    examples = []

    # Loop through rows
    for index, row in df.iterrows():
        examples.append((row['Text'], row['Label']))

    return examples


# Apply the function and create examples from dfs
labeled_examples = get_examples(df_train_for_ganbert)
unlabeled_examples= get_examples(df_unlabeled)
test_examples = get_examples(df_test)

In [12]:
print('Number of Labeled Datapoints:',len(labeled_examples))
print('Number of Unlabeled Datapoints:',len(unlabeled_examples))
print('Number of Test Datapoints:',len(test_examples))

Number of Labeled Datapoints: 35514
Number of Unlabeled Datapoints: 35513
Number of Test Datapoints: 3000


In [13]:
# Load tokenizer
model_name = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [14]:
# Main dataset function for creating semi-supervised dataset
def generate_data_loader(input_examples, label_masks, label_map, do_shuffle = False, balance_label_examples = False):

    examples = []

      # Count the percentage of labeled examples
    num_labeled_examples = 0
    for label_mask in label_masks:
        if label_mask:
            num_labeled_examples += 1
    label_mask_rate = num_labeled_examples/len(input_examples)

      # if required it applies the balance
    for index, ex in enumerate(input_examples):
        if label_mask_rate == 1 or not balance_label_examples:
            examples.append((ex, label_masks[index]))

    input_ids = []
    input_mask_array = []
    label_mask_array = []
    label_id_array = []

    # Tokenization
    for (text, label_mask) in tqdm.tqdm(examples):
        encoded_sent = tokenizer.encode(
            text[0], add_special_tokens=True, max_length=max_seq_length,
            padding="max_length", truncation=True)

        input_ids.append(encoded_sent)
        label_id_array.append(label_map[text[1]])
        label_mask_array.append(label_mask)

    # Attention to token
    for sent in input_ids:
        att_mask = [int(token_id > 0) for token_id in sent]
        input_mask_array.append(att_mask)

    # Convert to Tensor
    input_ids = torch.tensor(input_ids)
    input_mask_array = torch.tensor(input_mask_array)
    label_id_array = torch.tensor(label_id_array, dtype=torch.long)
    label_mask_array = torch.tensor(label_mask_array)

    # Make the TensorDataset
    dataset = TensorDataset(
          input_ids, input_mask_array, label_id_array, label_mask_array)
    
    # Shuffle for creating dataloader
    if do_shuffle:
        sampler = RandomSampler
    else:
        sampler = SequentialSampler

    # Make the DataLoader
    return DataLoader(
          dataset, sampler = sampler(dataset), batch_size = batch_size)

# A function for measure the time
def format_time(elapsed):

    # Round and convert to int
    elapsed_rounded = int(round((elapsed)))

    # Return time as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [15]:
# Define a label map for creating datasets
label_map = {}
for (i, label) in enumerate(label_list):
    label_map[label] = i


# Load trainset
train_examples = labeled_examples

# The labeled (train) dataset is assigned with a mask set to True
train_label_masks = np.ones(len(labeled_examples), dtype=bool)

# If unlabel examples are available
if unlabeled_examples:
     train_examples = train_examples + unlabeled_examples

   # The unlabeled (train) dataset is assigned with a mask set to False
     tmp_masks = np.zeros(len(unlabeled_examples), dtype=bool)
     train_label_masks = np.concatenate([train_label_masks,tmp_masks])

# Create trainloader
train_dataloader = generate_data_loader(train_examples, train_label_masks, label_map, do_shuffle = True, balance_label_examples = False)


# Load the test dataset
#The labeled (test) dataset is assigned with a mask set to True
test_label_masks = np.ones(len(test_examples), dtype=bool)

# Create test loader
test_dataloader = generate_data_loader(test_examples, test_label_masks, label_map, do_shuffle = False, balance_label_examples = False)

100%|██████████| 71027/71027 [01:38<00:00, 721.73it/s] 
/tmp/ipykernel_35/2232422918.py:42: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  label_mask_array = torch.tensor(label_mask_array)
100%|██████████| 3000/3000 [00:03<00:00, 876.80it/s] 


In [16]:
# Dataloader size
print(f'Number of training batchs with size of {batch_size}:',len(train_dataloader))

Number of training batchs with size of 32: 2220


## Part 4

In [18]:
# Create Generator class
class Generator(nn.Module):
    def __init__(self, noise_dim,hidden_layers, output_layer):
        super(Generator, self).__init__()
        self.Generator_Network=nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(512),
            nn.Dropout(p=0.1),
            nn.Linear(hidden_layers[0], output_layer))

    def forward(self,x):
        output = self.Generator_Network(x)
        return output

    
    
    

 # D with GRU   
class Discriminator(nn.Module):
    def __init__(self, input_dim, hidden_layers, output_layer):
        super(Discriminator, self).__init__()

        self.Discriminator_Features = nn.Sequential(
            nn.Dropout(p=0.1),
            nn.Linear(input_dim, hidden_layers[0]),
            nn.BatchNorm1d(hidden_layers[0]),
            nn.LeakyReLU(0.2),
            nn.Dropout(p=0.1))

        self.GRU = nn.GRU(hidden_layers[0], hidden_layers[1], batch_first=True)

        self.last_linear = nn.Linear(hidden_layers[1], output_layer)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        features = self.Discriminator_Features(x)
        gru_input = features.unsqueeze(1)  # Adding time dimension
        gru_output, _ = self.GRU(gru_input)
        gru_output = gru_output[:, -1, :]  # Taking the last output
        last_linear_val = self.last_linear(gru_output)
        output = self.softmax(last_linear_val)

        return features, last_linear_val, output

In [19]:
# Set hidden size for G/D
hidden_size = 512
hidden_layers_generator = [hidden_size,hidden_size//2, hidden_size//2, hidden_size]
hidden_levels_discriminator = [hidden_size,hidden_size//2, hidden_size//2, hidden_size]

# Set noise and output dimansions
noise_dim = 100
output_layer = 768

# Create G/D models
generator = Generator(
    noise_dim = noise_dim,
    hidden_layers = hidden_layers_generator,
    output_layer = output_layer)

discriminator = Discriminator(
    input_dim = output_layer,
    hidden_layers = hidden_levels_discriminator,
    output_layer = len(label_list) + 1)


# Load BERT
bert = AutoModel.from_pretrained(model_name)

# Put everything in the GPU if available
multi_gpu = True
if torch.cuda.is_available():
    generator.cuda()
    discriminator.cuda()
    bert.cuda()
#     if multi_gpu:
#         bert = torch.nn.DataParallel(bert)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [20]:
# Print G/D architectures
print(generator.parameters)
print('*' * 50)
print(discriminator.parameters)

<bound method Module.parameters of Generator(
  (Generator_Network): Sequential(
    (0): Linear(in_features=100, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Linear(in_features=256, out_features=512, bias=True)
    (4): LeakyReLU(negative_slope=0.2)
    (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): Dropout(p=0.1, inplace=False)
    (7): Linear(in_features=512, out_features=768, bias=True)
  )
)>
**************************************************
<bound method Module.parameters of Discriminator(
  (Discriminator_Features): Sequential(
    (0): Dropout(p=0.1, inplace=False)
    (1): Linear(in_features=768, out_features=512, bias=True)
    (2): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): LeakyReLU(negative_slope=0.2)
    (4): Dropout(p=0.1, inplace=False)
  )
  (GRU): GRU(51

In [21]:
# print(bert.parameters)

In [22]:
# torch.cuda.empty_cache()

In [23]:
# Set number of epochs
num_train_epochs = 10
multi_gpu = True
print_each_n_step = 200

# Save training/val/test stats
training_stats = []

# Measure the training time
total_t0 = time.time()

# Models parameters for passing to optimizer
bert_vars = [i for i in bert.parameters()]
d_vars = bert_vars + [v for v in discriminator.parameters()]
g_vars = [v for v in generator.parameters()]

# Set optimization parameters
learning_rate_discriminator = 5e-5
learning_rate_generator = 5e-5
epsilon = 1e-8

# Set optimizers
dis_optimizer = torch.optim.AdamW(d_vars, lr=learning_rate_discriminator)
gen_optimizer = torch.optim.AdamW(g_vars, lr=learning_rate_generator)

# Loop through epochs
for epoch_i in range(0, num_train_epochs):

    print("")
    print('===== Epoch {:} / {:} ====='.format(epoch_i + 1, num_train_epochs))
    print('Training...')

    # Measure epoch time
    t0 = time.time()

    # Reset the total loss for this epoch.
    tr_g_loss = 0
    tr_d_loss = 0

    # Set training mode
    bert.train()
    generator.train()
    discriminator.train()

    # Loop through the batches
    for step, batch in tqdm.tqdm(enumerate(train_dataloader)):

        # Progress update every print_each_n_step batches.
        if step % print_each_n_step == 0 and not step == 0:

            # Calculate time
            elapsed = format_time(time.time() - t0)

            # Print the progress
            print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(
                step, len(train_dataloader), elapsed))

        # Unpack the training batch
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        b_label_mask = batch[3].to(device)
        real_batch_size = b_input_ids.shape[0]

        # Encode real data in the BERT
        model_outputs = bert(b_input_ids, attention_mask=b_input_mask)
        hidden_states = model_outputs[-1]

        # Create noise to feed to the generator
        noise = torch.randn(
            real_batch_size, noise_dim, device=device)

        # Gnerate Fake data
        gen_rep = generator(noise)

        # Feed the output of the bert and the generator to disciminator
        disciminator_input = torch.cat([hidden_states, gen_rep], dim=0)

        # Get output of the disciminator
        features, logits, probs = discriminator(disciminator_input)

        # Separate the output of discriminatorfor the real and fake
        features_list = torch.split(features, real_batch_size)
        D_real_features = features_list[0]
        D_fake_features = features_list[1]

        logits_list = torch.split(logits, real_batch_size)
        D_real_logits = logits_list[0]
        D_fake_logits = logits_list[1]

        probs_list = torch.split(probs, real_batch_size)
        D_real_probs = probs_list[0]
        D_fake_probs = probs_list[1]

        # Generator LOSS
        g_loss_d = -1 * torch.mean(torch.log(1 - D_fake_probs[:,-1] + epsilon))

        g_feat_reg = torch.mean(torch.pow(
            torch.mean(D_real_features, dim=0)
             - torch.mean(D_fake_features, dim=0), 2))

        g_loss = g_loss_d + g_feat_reg

        # Disciminator LOSS
        logits = D_real_logits[:,0:-1]
        log_probs = F.log_softmax(logits, dim=-1)

        # The Loss for unlabeled data is masked out
        label2one_hot = torch.nn.functional.one_hot(b_labels, len(label_list))

        per_example_loss = -torch.sum(label2one_hot * log_probs, dim=-1)

        per_example_loss = torch.masked_select(
            per_example_loss, b_label_mask.to(device))

        labeled_example_count = per_example_loss.type(torch.float32).numel()

        if labeled_example_count == 0:
            D_L_Supervised = 0

        else:
            D_L_Supervised = torch.div(
              torch.sum(per_example_loss.to(device)), labeled_example_count)

        D_L_unsupervised1U = -1 * torch.mean(
            torch.log(1 - D_real_probs[:, -1] + epsilon))

        D_L_unsupervised2U = -1 * torch.mean(
            torch.log(D_fake_probs[:, -1] + epsilon))

        d_loss = D_L_Supervised + D_L_unsupervised1U + D_L_unsupervised2U

        # Reset gradients
        gen_optimizer.zero_grad()
        dis_optimizer.zero_grad()

        # Backward pass
        g_loss.backward(retain_graph=True)
        d_loss.backward()

        # Update weights
        gen_optimizer.step()
        dis_optimizer.step()

        # Save the losses to report
        tr_g_loss += g_loss.item()
        tr_d_loss += d_loss.item()


    # Calculate the average loss over the batches.
    avg_train_loss_g = tr_g_loss / len(train_dataloader)
    avg_train_loss_d = tr_d_loss / len(train_dataloader)

    # Measure epoch time
    training_time = format_time(time.time() - t0)

    # Pritn stats
    print("")
    print("Average training loss generetor: {0:.3f}".format(avg_train_loss_g))
    print("Average training loss discriminator: {0:.3f}".format(avg_train_loss_d))
    print("Training epcoh took: {:}".format(training_time))


    # Test
    print("")
    print("Running Test...")

    t0 = time.time()

    # Set validation mode
    bert.eval()
    discriminator.eval()
    generator.eval()

    # Tracking variables
    total_test_accuracy = 0

    total_test_loss = 0
    nb_test_steps = 0

    all_preds = []
    all_labels_ids = []

    # Define Loss function
    nll_loss = torch.nn.CrossEntropyLoss(ignore_index=-1)

    # Loop through test patches
    for batch in test_dataloader:

        # Unpack the test batch
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        # No grad block
        with torch.no_grad():
            model_outputs = bert(
                b_input_ids, attention_mask=b_input_mask)

            hidden_states = model_outputs[-1]
            _, logits, probs = discriminator(hidden_states)

            filtered_logits = logits[:,0:-1]

            # Accumulate the test loss.
            total_test_loss += nll_loss(filtered_logits, b_labels)

        # Accumulate the predictions and the input labels
        _, preds = torch.max(filtered_logits, 1)
        all_preds += preds.detach().cpu()
        all_labels_ids += b_labels.detach().cpu()

    # Report the final accuracy for this validation run.
    all_preds = torch.stack(all_preds).numpy()
    all_labels_ids = torch.stack(all_labels_ids).numpy()
    test_accuracy = np.sum(all_preds == all_labels_ids) / len(all_preds)
    print(" Test Accuracy: {0:.3f}".format(test_accuracy))

    # Calculate the average loss over all of the batches.
    avg_test_loss = total_test_loss / len(test_dataloader)
    avg_test_loss = avg_test_loss.item()

    # Measure validation runtime
    test_time = format_time(time.time() - t0)

    # Print validation stats
    print("  Test Loss: {0:.3f}".format(avg_test_loss))
    print("  Test took: {:}".format(test_time))

    # Store stats from this epoch.
    training_stats.append(
        {'epoch': epoch_i + 1, 'Training Loss generator': avg_train_loss_g,
         'Training Loss discriminator': avg_train_loss_d,
         'Valid. Loss': avg_test_loss, 'Valid. Accur.': test_accuracy,
         'Training Time': training_time, 'Test Time': test_time})


===== Epoch 1 / 10 =====
Training...


200it [04:02,  1.21s/it]

  Batch   200  of  2,220.    Elapsed: 0:04:02.


400it [08:04,  1.21s/it]

  Batch   400  of  2,220.    Elapsed: 0:08:04.


...

  Batch 2,000  of  2,220.    Elapsed: 0:40:34.


2200it [44:37,  1.21s/it]

  Batch 2,200  of  2,220.    Elapsed: 0:44:37.


2220it [45:01,  1.22s/it]



Average training loss generetor: 0.702
Average training loss discriminator: 0.757
Training epcoh took: 0:45:01

Running Test...
 Test Accuracy: 0.567
  Test Loss: 2.384
  Test took: 0:00:23


In [26]:
for stat in training_stats:
    print(stat)

print("\nTraining complete!")
print("Total training took {:} (h:mm:ss)".format(format_time(time.time()-total_t0)))

{'epoch': 1, 'Training Loss generator': 0.7266803198301041, 'Training Loss discriminator': 1.3592393261892302, 'Valid. Loss': 1.1866892576217651, 'Valid. Accur.': 0.6446666666666667, 'Training Time': '0:44:50', 'Test Time': '0:00:23'}
{'epoch': 2, 'Training Loss generator': 0.7185232083271216, 'Training Loss discriminator': 0.9966905983718666, 'Valid. Loss': 1.608597993850708, 'Valid. Accur.': 0.5966666666666667, 'Training Time': '0:44:55', 'Test Time': '0:00:23'}
{'epoch': 3, 'Training Loss generator': 0.7125020408684068, 'Training Loss discriminator': 0.8885354578495026, 'Valid. Loss': 1.3951901197433472, 'Valid. Accur.': 0.6446666666666667, 'Training Time': '0:45:00', 'Test Time': '0:00:23'}
{'epoch': 4, 'Training Loss generator': 0.709223502605885, 'Training Loss discriminator': 0.8403594483394881, 'Valid. Loss': 1.9797426462173462, 'Valid. Accur.': 0.5796666666666667, 'Training Time': '0:44:59', 'Test Time': '0:00:23'}
{'epoch': 5, 'Training Loss generator': 0.7074057318605819, 'T

In [27]:
# Save training and validation/test stats to report later
# and comprasion with other parts

# Write stats as a json file
with open("Part4_Best_stats.json", "w") as outfile:
     json.dump(training_stats, outfile)
print('Saved')

Saved
